[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/07_computer_vision/project.ipynb)

# Project 07 — Nhận dạng hình khối và "học đường tắt" (Computer Vision, CNN)

> Một dây chuyền phân loại sản phẩm dùng camera nhận dạng **hình dạng** chi tiết (tròn, vuông, tam giác, chữ thập, vành khuyên). Trong dữ liệu thu thập ở nhà máy cũ, mỗi loại chi tiết **gần như luôn được sơn một màu riêng** (95%). Nhà máy mới sơn màu **ngẫu nhiên**. Mô hình phải nhận dạng **hình**, không phải **màu**.

**Bạn sẽ làm**:
1. Xem dữ liệu; tách tập kiểm tra **cùng phân phối** (nhà máy cũ) và **khác phân phối** (nhà máy mới, màu ngẫu nhiên).
2. Baseline "chỉ nhìn màu" — chứng minh đường tắt tồn tại.
3. Huấn luyện CNN trên ảnh màu; đo độ chính xác ở cả hai tập kiểm tra; chẩn đoán.
4. Hai cách sửa: ảnh xám chuẩn hoá theo từng ảnh; tăng cường dữ liệu xáo trộn màu.
5. Ma trận nhầm lẫn, báo cáo.

**Tiêu chí đạt**: baseline màu đạt > 85% ở nhà máy cũ nhưng < 35% ở nhà máy mới · CNN ảnh màu: > 95% ở nhà máy cũ và tụt > 15 điểm ở nhà máy mới · ít nhất một cách sửa đạt > 88% ở nhà máy mới với chênh lệch < 6 điểm.

**Chương liên quan**: 29 (CNN), 28 (BatchNorm, lịch learning rate), 33 (tăng cường dữ liệu, dịch chuyển miền), 05 (phân phối dịch chuyển), 36 (đường tắt, giải thích). Cần `pip install -e ".[dl]"`.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/07_computer_vision

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import SHAPE_CLASSES, make_shapes

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError('Project này cần PyTorch: pip install -e ".[dl]"') from e

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = SHAPE_CLASSES
X_train, y_train = make_shapes(6000, 28, classes=CLASSES, rgb=True, color_bias=0.95, seed=0)
X_old, y_old = make_shapes(2000, 28, classes=CLASSES, rgb=True, color_bias=0.95, seed=2)     # nhà máy cũ
X_new, y_new = make_shapes(2000, 28, classes=CLASSES, rgb=True, color_bias=0.0, seed=3)      # nhà máy mới
print(X_train.shape, "| lớp:", CLASSES, "| thiết bị:", device)
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for j in range(8):
    axes[0, j].imshow(X_old[j])
    axes[0, j].set_title(CLASSES[y_old[j]], fontsize=8)
    axes[1, j].imshow(X_new[j])
    axes[1, j].set_title(CLASSES[y_new[j]], fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-12, 14, "cũ", fontsize=10)
axes[1, 0].text(-12, 14, "mới", fontsize=10)
plt.show()

## Bước 1 — Baseline "chỉ nhìn màu"

Đặc trưng: **màu trung bình của các pixel sáng** (độ sáng trung bình 3 kênh > 0.35 — phần hình, không phải nền) → 3 số mỗi ảnh. Viết `color_features(X)` → mảng (n, 3); huấn luyện `LogisticRegression(max_iter=1000)` trên train → `color_acc_old`, `color_acc_new`.

In [ ]:
def color_features(X):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return np.zeros((len(X), 3))
    bright = X.mean(axis=3) > 0.35
    counts = bright.sum(axis=(1, 2))[:, None] + 1e-6
    return (X * bright[..., None]).sum(axis=(1, 2)) / counts
    ### END CODE HERE ###


### START CODE HERE ### (≈ 3 dòng)
color_model = LogisticRegression(max_iter=1000).fit(color_features(X_train), y_train)
color_acc_old = color_model.score(color_features(X_old), y_old)
color_acc_new = color_model.score(color_features(X_new), y_new)
### END CODE HERE ###
print(f"chỉ nhìn màu — nhà máy cũ: {color_acc_old:.1%} | nhà máy mới: {color_acc_new:.1%} (ngẫu nhiên: {1 / len(CLASSES):.0%})")

In [ ]:
check_equal(color_features(X_train[:4]).shape, (4, 3), "3 đặc trưng màu")
check_greater(color_acc_old, 0.85, "màu gần như đủ để phân loại ở nhà máy cũ")
check_less(color_acc_new, 0.35, "màu vô dụng ở nhà máy mới")

Một mô hình 3 tham số mỗi lớp đã đạt độ chính xác rất cao mà **không nhìn hình dạng**. Một CNN sẽ học đường tắt dễ nhất có thể — trừ khi ta ngăn nó lại.

## Bước 2 — CNN trên ảnh màu

Hàm huấn luyện cho sẵn (BatchNorm, Adam + OneCycle, 12 epoch). Viết `make_cnn(in_channels)`: 3 khối `Conv2d(3×3, padding 1) → BatchNorm2d → ReLU` với 32, 64, 128 kênh; `MaxPool2d(2)` sau khối 1 và 2; `AdaptiveAvgPool2d(1)` → `Flatten` → `Linear(128, 5)`. Huấn luyện trên ảnh màu → `rgb_model`; đo `rgb_acc_old`, `rgb_acc_new`.

In [ ]:
def to_tensor(X):
    return torch.tensor(X).permute(0, 3, 1, 2)


def train(model, X, y, epochs=12, augment=None, seed=0):
    torch.manual_seed(seed)
    model = model.to(device)
    y = torch.tensor(y)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    steps = epochs * ((len(X) + 127) // 128)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=steps)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 128):
            idx = perm[i:i + 128]
            xb = X[idx] if augment is None else augment(X[idx])
            loss = nn.functional.cross_entropy(model(xb.to(device)), y[idx].to(device))
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
    return model


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 512].to(device)).argmax(1).cpu() for i in range(0, len(X), 512)]).numpy()


def make_cnn(in_channels):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return nn.Sequential(nn.Flatten(), nn.Linear(in_channels * 28 * 28, len(CLASSES)))
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, len(CLASSES)))
    ### END CODE HERE ###


t0 = time.perf_counter()
### START CODE HERE ### (≈ 3 dòng)
rgb_model = train(make_cnn(3), to_tensor(X_train), y_train)
rgb_acc_old = (predict(rgb_model, to_tensor(X_old)) == y_old).mean()
rgb_acc_new = (predict(rgb_model, to_tensor(X_new)) == y_new).mean()
### END CODE HERE ###
print(f"CNN ảnh màu — nhà máy cũ: {rgb_acc_old:.1%} | nhà máy mới: {rgb_acc_new:.1%} ({time.perf_counter() - t0:.0f} giây)")

In [ ]:
check_equal(make_cnn(3)(torch.zeros(2, 3, 28, 28)).shape, (2, 5), "đầu ra 5 lớp")
check_greater(rgb_acc_old, 0.95, "rất tốt ở nhà máy cũ")
check_greater(rgb_acc_old - rgb_acc_new, 0.15, "tụt mạnh ở nhà máy mới — đã học một phần đường tắt màu")

## Bước 3 — Chẩn đoán

Ở nhà máy mới, khi CNN **sai**, nó có đoán theo **màu** không? Với mỗi ảnh nhà máy mới bị đoán sai, lấy lớp mà baseline màu dự đoán; tính tỉ lệ trùng với dự đoán của CNN → `agree_with_color`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
pred_new = predict(rgb_model, to_tensor(X_new))
wrong = pred_new != y_new
color_pred_new = color_model.predict(color_features(X_new))
agree_with_color = (pred_new[wrong] == color_pred_new[wrong]).mean()
### END CODE HERE ###
print(f"{wrong.mean():.1%} ảnh nhà máy mới bị đoán sai; trong số đó {agree_with_color:.0%} trùng với dự đoán 'chỉ nhìn màu' (ngẫu nhiên ~{1 / (len(CLASSES) - 1):.0%})")

In [ ]:
check_greater(agree_with_color, 0.35, "lỗi trùng với dự đoán theo màu nhiều hơn hẳn mức ngẫu nhiên (~25%)")

## Bước 4 — Hai cách sửa

**(A) Bỏ thông tin màu khỏi đầu vào**: chuyển xám (trung bình 3 kênh) rồi **chuẩn hoá từng ảnh** về [0, 1] (trừ min, chia max) — nếu không, độ sáng của mỗi màu vẫn rò rỉ thông tin lớp. Viết `to_gray(X)` → tensor (n, 1, 28, 28) → `gray_model`.

**(B) Giữ màu nhưng làm nó vô nghĩa khi huấn luyện**: tăng cường dữ liệu — **xáo thứ tự kênh** màu và nhân mỗi kênh với hệ số ngẫu nhiên trong [0.4, 1.4] (cắt về [0, 1]). Viết `color_jitter(batch)` → `jitter_model`.

Đo độ chính xác ở hai nhà máy → `fixes` (DataFrame-like dict: tên → (cũ, mới)).

In [ ]:
def to_gray(X):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return to_tensor(X)[:, :1]
    g = to_tensor(X).mean(1, keepdim=True)
    g = g - g.amin(dim=(2, 3), keepdim=True)
    return g / (g.amax(dim=(2, 3), keepdim=True) + 1e-6)
    ### END CODE HERE ###


def color_jitter(batch):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return batch
    shuffled = batch[:, torch.randperm(3)]
    return (shuffled * (torch.rand(len(batch), 3, 1, 1) + 0.4)).clamp(0, 1)
    ### END CODE HERE ###


t0 = time.perf_counter()
### START CODE HERE ### (≈ 5 dòng)
# STARTER: gray_model = None     # train(make_cnn(1), to_gray(X_train), y_train)
# STARTER: jitter_model = None   # train(make_cnn(3), ..., augment=color_jitter)
# STARTER: fixes = {}
gray_model = train(make_cnn(1), to_gray(X_train), y_train)
jitter_model = train(make_cnn(3), to_tensor(X_train), y_train, augment=color_jitter)
fixes = {"(A) ảnh xám chuẩn hoá": ((predict(gray_model, to_gray(X_old)) == y_old).mean(), (predict(gray_model, to_gray(X_new)) == y_new).mean()),
         "(B) xáo trộn màu": ((predict(jitter_model, to_tensor(X_old)) == y_old).mean(), (predict(jitter_model, to_tensor(X_new)) == y_new).mean())}
### END CODE HERE ###
print(f"{'CNN ảnh màu (gốc)':24s}: cũ {rgb_acc_old:.1%} | mới {rgb_acc_new:.1%}")
for name, (old, new) in fixes.items():
    print(f"{name:24s}: cũ {old:.1%} | mới {new:.1%}")
print(f"({time.perf_counter() - t0:.0f} giây)")

In [ ]:
gray_batch = to_gray(X_train[:8])
check_equal(tuple(gray_batch.shape), (8, 1, 28, 28), "ảnh xám 1 kênh")
check_close(gray_batch.amax(dim=(1, 2, 3)).numpy(), np.ones(8), tol=1e-4, message="mỗi ảnh chuẩn hoá về max = 1")
check_true(any(new > 0.88 and old - new < 0.06 for old, new in fixes.values()), "ít nhất một cách sửa bền vững ở nhà máy mới")

## Bước 5 — Mô hình tốt nhất: nhầm lẫn ở đâu?

Chọn cách sửa có độ chính xác nhà máy mới cao nhất; vẽ ma trận nhầm lẫn trên nhà máy mới.

In [ ]:
best_name = max(fixes, key=lambda k: fixes[k][1])
best_model, prep = (gray_model, to_gray) if best_name.startswith("(A)") else (jitter_model, to_tensor)
ConfusionMatrixDisplay.from_predictions(y_new, predict(best_model, prep(X_new)), display_labels=CLASSES, cmap="Blues", xticks_rotation=30)
plt.title(f"{best_name} — nhà máy mới")
plt.show()

**Nhận xét mẫu:** Nhầm lẫn còn lại chủ yếu giữa **tròn ↔ vành khuyên** (vành khuyên nhỏ, nhiễu cao trông như tròn đặc) và **vuông ↔ tam giác/chữ thập** khi hình nhỏ, xoay. Đây là lỗi về **hình dạng thật**, đúng loại lỗi ta chấp nhận được và cải thiện bằng ảnh độ phân giải cao hơn hoặc nhiều dữ liệu hơn ở kích thước nhỏ.

## Bước 6 — Báo cáo

Viết 5–8 câu cho kỹ sư dây chuyền: điều gì đã xảy ra với mô hình đầu tiên, vì sao độ chính xác validation "đẹp" không đủ, hai cách sửa và đánh đổi (bỏ màu vĩnh viễn vs tăng cường dữ liệu), khuyến nghị (thu ảnh từ nhà máy mới để kiểm tra trước khi triển khai, giám sát theo nhà máy), và bài học chung về đường tắt.

✍️ Báo cáo:

**Báo cáo mẫu:** Mô hình CNN đầu tiên đạt ~98% trên dữ liệu nhà máy cũ nhưng chỉ ~70% ở nhà máy mới: vì ở nhà máy cũ màu sơn gần như xác định loại chi tiết, mô hình học một phần "nhìn màu" thay vì "nhìn hình" — gần một nửa số lỗi ở nhà máy mới trùng với dự đoán của một mô hình chỉ dùng màu (gấp gần 2 lần mức ngẫu nhiên). Validation lấy từ cùng nhà máy cũ **không thể** phát hiện điều này. Hai cách sửa đều đưa độ chính xác ở nhà máy mới lên ~92–95% và gần bằng nhà máy cũ: (A) chỉ dùng ảnh xám đã chuẩn hoá độ sáng — đơn giản, loại hẳn thông tin màu (phù hợp nếu màu không bao giờ có ích); (B) giữ ảnh màu nhưng xáo trộn màu khi huấn luyện — linh hoạt hơn nếu sau này màu mang thông tin hợp lệ. Khuyến nghị: trước mỗi lần triển khai ở môi trường mới, thu một mẫu ảnh có nhãn từ môi trường đó để kiểm tra; báo cáo độ chính xác **theo từng nhà máy**; khi dữ liệu huấn luyện có thuộc tính tương quan giả với nhãn (màu, nền, góc chụp, logo), chủ động phá vỡ tương quan đó bằng thu thập hoặc tăng cường dữ liệu.